In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: функція Qiskit від Q-CTRL Fire Opal
*Дивись [довідник API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Qiskit Functions — це експериментальна функція, доступна лише користувачам IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вона перебуває у статусі попереднього релізу та може змінюватись.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Огляд
За допомогою Fire Opal Optimization Solver ти можеш розв'язувати задачі оптимізації утилітарного масштабу на квантовому залізі без потреби в квантовій експертизі. Просто введи визначення задачі на високому рівні — і Solver зробить усе інше. Весь робочий процес є шумостійким і використовує [Fire Opal Performance Management](/guides/q-ctrl-performance-management) «під капотом». Solver стабільно надає точні рішення для класично складних задач навіть у повному масштабі на найбільших QPU від IBM&reg;.

Solver є гнучким і може розв'язувати комбінаторні задачі оптимізації, визначені як цільові функції або довільні графи. Задачі не потрібно відображати на топологію пристрою. Розв'язуються як необмежені, так і обмежені задачі, за умови що обмеження можуть бути сформульовані у вигляді штрафних доданків. Приклади, наведені в цьому посібнику, демонструють, як розв'язати необмежену та обмежену задачу оптимізації утилітарного масштабу, використовуючи різні типи вхідних даних для Solver. Перший приклад охоплює задачу max-cut на 156-вузловому 3-регулярному графі, тоді як другий — задачу Minimum Vertex Cover на 50 вузлах, визначену через цільову функцію.

Щоб отримати доступ до Optimization Solver, [звернись до Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Опис функції
Solver повністю оптимізує та автоматизує весь алгоритм — від придушення помилок на рівні заліза до ефективного відображення задачі та замкненої класичної оптимізації. «За лаштунками» конвеєр Solver зменшує помилки на кожному етапі, забезпечуючи підвищену продуктивність, необхідну для реального масштабування. Базовий робочий процес натхненний алгоритмом квантової наближеної оптимізації (QAOA) — гібридним квантово-класичним алгоритмом. Детальний опис повного робочого процесу Optimization Solver наведено в [опублікованій статті](https://arxiv.org/abs/2406.01743).

![Візуалізація робочого процесу Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Щоб розв'язати загальну задачу за допомогою Optimization Solver:
1. Визнач свою задачу як цільову функцію, граф або `SparsePauliOp` спінового ланцюжка.
2. Підключись до функції через Qiskit Functions Catalog.
3. Запусти задачу на Solver і отримай результати.
### Підтримувані формати задач
- Представлення цільової функції у вигляді поліноміального виразу. В ідеалі — створений у Python на основі існуючого об'єкта SymPy Poly та відформатований у рядок за допомогою [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Представлення задачі у вигляді графа для конкретного типу задачі. Граф слід створювати за допомогою бібліотеки networkx у Python, а потім перетворювати на рядок функцією networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Представлення задачі у вигляді спінового ланцюжка. Спіновий ланцюжок має бути представлений як об'єкт `SparsePauliOp`; детальніше — у [документації](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

> **Note:** Якщо ти хочеш використовувати бекенд, який ця функція наразі не підтримує, [звернись до Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com), щоб додати підтримку.
## Бенчмарки
[Опубліковані результати бенчмаркінгу](https://arxiv.org/abs/2406.01743) показують, що Solver успішно розв'язує задачі на понад 120 кубітах, навіть перевершуючи раніше опубліковані результати на пристроях квантового відпалу та пастки іонів. Наступні метрики бенчмарку дають приблизне уявлення про точність і масштабування типів задач на основі кількох прикладів. Фактичні метрики можуть відрізнятись залежно від різних характеристик задачі, таких як кількість доданків у цільовій функції (щільність) та їхня локальність, кількість змінних і порядок полінома.

«Кількість кубітів», що вказана, не є жорстким обмеженням, а лише приблизними порогами, при яких можна очікувати надзвичайно стабільної точності рішення. Задачі більшого розміру вже були успішно розв'язані, і тестування за межами цих порогів заохочується.

Довільне з'єднання кубітів підтримується для всіх типів задач.

| Тип задачі    | Кількість кубітів | Приклад | Точність | Загальний час (с) | Використання Runtime (с) | Кількість ітерацій
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Розріджено-зв'язані квадратичні задачі  | 156 | 3-регулярний max-cut| 100%     | 1764     | 293          | 16 |
| Бінарна оптимізація вищого порядку | 156 | Ізингова модель спінового скла | 100%      | 1461     | 272          | 16 |
| Щільно-зв'язані квадратичні задачі | 50 | Повністю-зв'язаний max-cut| 100%      |  1758    | 268  | 12 |
| Обмежена задача зі штрафними доданками | 50 | Зважений Minimum Vertex Cover із щільністю ребер 8% | 100%      | 1074     | 215 | 10 |
## Початок роботи
Спочатку пройди автентифікацію за допомогою свого [API-ключа IBM Quantum](http://quantum.cloud.ibm.com/). Потім вибери функцію Qiskit наступним чином. (Цей фрагмент передбачає, що ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Визнач задачу
Ти можеш запустити задачу Max-Cut, визначивши задачу у вигляді графа та вказавши `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Запусти задачу
При використанні графового методу вводу вкажи тип задачі.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. Отримай результат
Отримай оптимальне значення розрізу зі словника результатів.

> **Note:** Відображення змінних на бітовий рядок може змінитись. Вихідний словник містить підсловник `variables_to_bitstring_index_map`, який допомагає перевірити порядок.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

Ти можеш перевірити точність результату, розв'язавши задачу класично за допомогою відкритих солверів, таких як [PuLP](https://coin-or.github.io/pulp/), якщо граф не є щільно зв'язаним. Для задач із високою щільністю може знадобитись використання просунутих класичних солверів для підтвердження рішення.
## Приклад: обмежена оптимізація
Попередній приклад max-cut є класичною квадратичною необмеженою задачею бінарної оптимізації. Optimization Solver від Q-CTRL може використовуватись для різних типів задач, зокрема обмеженої оптимізації. Ти можеш розв'язувати довільні типи задач, подаючи визначення задачі у вигляді полінома, де обмеження моделюються як штрафні доданки.

Наступний приклад демонструє, як побудувати цільову функцію для обмеженої задачі оптимізації — [мінімального покриття вершин](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Окрім пакетів `qiskit-ibm-catalog` і `qiskit`, для виконання цього прикладу тобі знадобляться такі пакети: `numpy`, `networkx` і `sympy`. Ти можеш встановити їх, розкоментувавши наступну комірку, якщо запускаєш цей приклад у ноутбуці з ядром IPython.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Визнач задачу
Визнач випадкову задачу MVC, згенерувавши граф із вузлами, що мають випадкові ваги.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![Вихід попередньої комірки коду](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Стандартна модель оптимізації для зваженого MVC може бути сформульована наступним чином. Спочатку необхідно додати штраф для будь-якого випадку, коли ребро не з'єднане з вершиною з підмножини. Тому нехай $n_i = 1$, якщо вершина $i$ входить до покриття (тобто знаходиться у підмножині), і $n_i = 0$ в іншому випадку. По-друге, мета полягає у мінімізації загальної кількості вершин у підмножині, що може бути виражено такою функцією:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Тепер кожне ребро в графі повинне мати хоча б одну кінцеву точку з покриття, що можна виразити як нерівність:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Будь-який випадок, коли ребро не з'єднане з вершиною покриття, має бути оштрафований. Це можна представити у цільовій функції, додавши штрафний доданок виду $P(1-n_i-n_j+n_i n_j)$, де $P$ — це позитивна штрафна константа. Таким чином, необмежена альтернатива обмеженій нерівності для зваженого MVC виглядає так:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Запусти задачу

In [17]:
print(mvc_job.status())

QUEUED


Перевір [статус](/guides/functions#check-job-status) робочого навантаження своєї функції Qiskit або отримай [результати](/guides/functions#retrieve-results) наступним чином:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>